#Скачивание фотографий с Циана

Фото сохраняются в папку photos_new (`photos_new/<id объявления>/1.jpg, 2.jpg, ...`)

In [4]:
import os
import json
import urllib.request
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

df = pd.read_csv('photos_to_keep_triple_sex.csv' , encoding='utf-8')
rows = df.to_dict('records')[:10] #лимит для тестирования

Данную ячейку можно перезапускать для докачивания отвалившихся фото

In [5]:
spis = []
skip = 0
for r in rows:
    id = str(r['id'])
    links = r['photos_to_keep']
    if links:
        links = json.loads(links)
    else:
        skip += 1
        continue

    folder = os.path.join('photos_new', id)
    os.makedirs(folder, exist_ok=True)

    for k, u in enumerate(links, start=1):
        p = os.path.join(folder, f'{k}.jpg')
        if not os.path.exists(p):
            spis.append([u, p])
print(f'Фото для скачивания: {len(spis)}, пропущенно квартир без фото: {skip}')

def download_photo(one):
    req = urllib.request.Request(one[0], headers={'User-Agent': 'Mozilla/5.0', 'Referer': 'https://www.cian.ru/'})
    try:
        with urllib.request.urlopen(req, timeout=10) as r:
            photo = r.read()
        with open(one[1], 'wb') as put:
            put.write(photo)
        return True
    except:
        return False

#скачиваем с помощью многопоточности
c = 0
ers = 0

with ThreadPoolExecutor(max_workers=16) as ex:
    for ok in ex.map(download_photo, spis):
        c += 1
        if not ok:
            ers += 1
        if c % 500 == 0:
            print(f'Обработано {c} из {len(spis)}, ошибок: {ers}')

print(f'Скачали {c - ers} фотографий из {len(spis)}, ошибок: {ers}')

Фото для скачивания: 96, пропущенно квартир без фото: 0
Скачали 95 фотографий из 96, ошибок: 1
